# Stage 2: Data Collection Pipeline
This notebook runs the metadata, transcript, and comment scrapers to build the raw database storage.

In [ ]:
import sys
import os
from pathlib import Path
import json

import sys, os
from pathlib import Path
cwd = Path(os.getcwd()).resolve()
base_dir = cwd if (cwd / 'config').exists() else (cwd.parent if (cwd.parent / 'config').exists() else cwd)
if str(base_dir) not in sys.path: sys.path.insert(0, str(base_dir))

from config import settings
from src.scraping.metadata_scraper import scrape_metadata_for_playlist
from src.scraping.transcript_scraper import scrape_transcripts_for_playlist
from src.scraping.comment_scraper import scrape_comments_for_playlist

## Check environment variables
We need `YOUTUBE_API_KEY` to collect metadata and comments.

In [ ]:
if not settings.YOUTUBE_API_KEY:
    print("WARNING: YOUTUBE_API_KEY is not defined in settings / .env.")
    print("Generating mock dataset files in raw/ directory to enable verification of downstream steps.")
    
    # Generate mock metadata, transcripts, and comments for 10 videos
    for i in range(1, 11):
        vid = f"video_mock_{i}"
        
        # 1. Metadata Mock
        meta_file = settings.RAW_METADATA_DIR / f"{vid}.json"
        if not meta_file.exists():
            meta_payload = {
                "id": vid,
                "snippet": {
                    "title": f"Belajar Sains Kok Bisa Part {i}",
                    "description": f"Kok Bisa video explaining interesting science concepts, Episode {i}.",
                    "publishedAt": "2026-01-01T00:00:00Z"
                },
                "statistics": {
                    "viewCount": str(10000 * i),
                    "likeCount": str(500 * i),
                    "commentCount": str(50 * i)
                }
            }
            with open(meta_file, "w") as f:
                json.dump(meta_payload, f, indent=4)
                
        # 2. Transcript Mock
        trans_file = settings.RAW_TRANSCRIPTS_DIR / f"{vid}.json"
        if not trans_file.exists():
            trans_payload = {
                "video_id": vid,
                "language_code": "id",
                "is_generated": True,
                "transcript": [
                    {"text": "Halo teman-teman!", "start": 0.0, "duration": 2.0},
                    {"text": "Kembali lagi di Kok Bisa.", "start": 2.0, "duration": 3.0},
                    {"text": "Hari ini kita akan membahas tentang sains.", "start": 5.0, "duration": 4.0},
                    {"text": "Mengapa air laut itu asin?", "start": 9.0, "duration": 3.0},
                    {"text": "Terima kasih sudah menonton!", "start": 12.0, "duration": 3.0}
                ]
            }
            with open(trans_file, "w") as f:
                json.dump(trans_payload, f, indent=4)
                
        # 3. Comment Mock
        comm_file = settings.RAW_COMMENTS_DIR / f"{vid}.json"
        if not comm_file.exists():
            comm_payload = [
                {
                    "comment_id": f"c_{vid}_1",
                    "author_name": "Andi",
                    "author_channel_id": "ch_andi",
                    "text": "Keren sekali penjelasannya! Sangat mudah dipahami.",
                    "like_count": 10,
                    "published_at": "2026-01-02T10:00:00Z",
                    "updated_at": "2026-01-02T10:00:00Z",
                    "replies": [
                        {
                            "comment_id": f"c_{vid}_1_r1",
                            "author_name": "Budi",
                            "author_channel_id": "ch_budi",
                            "text": "Setuju! Saya juga suka bagian visualnya.",
                            "like_count": 2,
                            "published_at": "2026-01-02T11:00:00Z",
                            "updated_at": "2026-01-02T11:00:00Z"
                        }
                    ]
                },
                {
                    "comment_id": f"c_{vid}_2",
                    "author_name": "Caca",
                    "author_channel_id": "ch_caca",
                    "text": "Apakah ada hubungannya dengan garam di darat?",
                    "like_count": 5,
                    "published_at": "2026-01-02T12:00:00Z",
                    "updated_at": "2026-01-02T12:00:00Z",
                    "replies": []
                },
                {
                    "comment_id": f"c_{vid}_3",
                    "author_name": "Bot123",
                    "author_channel_id": "ch_bot",
                    "text": "Visit my website https://spam-link.com for free followers!",
                    "like_count": 0,
                    "published_at": "2026-01-02T13:00:00Z",
                    "updated_at": "2026-01-02T13:00:00Z",
                    "replies": []
                }
            ]
            with open(comm_file, "w") as f:
                json.dump(comm_payload, f, indent=4)
    print("Mock dataset files populated successfully.")
else:
    print("API Key found. Executing real scraping pipeline...")
    scrape_metadata_for_playlist()
    scrape_transcripts_for_playlist()
    scrape_comments_for_playlist()